## ✅ Preflight Check
Run this cell first — it validates your environment before anything else executes.

In [ ]:
import os, shutil, sys, subprocess

PASS = "\033[92m✔\033[0m"
FAIL = "\033[91m✘\033[0m"
WARN = "\033[93m⚠\033[0m"
all_ok = True

def ok(msg):   print(f"  {PASS}  {msg}")
def fail(msg): print(f"  {FAIL}  {msg}"); global all_ok; all_ok = False
def warn(msg): print(f"  {WARN}  {msg}")

print("=" * 55)
print("  LiteLDM — Environment Preflight Check")
print("=" * 55)

# ── 1. Python version ────────────────────────────────────────
print("\n[1] Python")
vi = sys.version_info
if vi.major == 3 and vi.minor >= 9:
    ok(f"Python {vi.major}.{vi.minor}.{vi.micro}")
else:
    fail(f"Python {vi.major}.{vi.minor} — need 3.9+")

# ── 2. CUDA / GPU ────────────────────────────────────────────
print("\n[2] GPU / CUDA")
try:
    import torch
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        ok(f"CUDA available — {gpu}  ({vram:.1f} GB VRAM)")
        if vram < 8:
            warn("Less than 8 GB VRAM — consider reducing batch size to 4")
        elif vram < 12:
            warn("8–12 GB VRAM — reduce VAE batch size to 8 if OOM")
        cuda_ver = torch.version.cuda
        ok(f"CUDA version: {cuda_ver}")
    else:
        fail("CUDA not available — training will be extremely slow on CPU")
except ImportError:
    fail("PyTorch not installed — run the pip install cell first")

# ── 3. $LOCAL ────────────────────────────────────────────────
print("\n[3] $LOCAL storage")
LOCAL = os.environ.get("LOCAL", "")
if LOCAL:
    ok(f"$LOCAL is set → {LOCAL}")
    free = shutil.disk_usage(LOCAL).free / 1e9
    if free >= 10:
        ok(f"Free space on $LOCAL: {free:.1f} GB")
    else:
        warn(f"Only {free:.1f} GB free on $LOCAL — dataset may not fit")
else:
    fail("$LOCAL is not set. Run: export LOCAL=/path/to/local/storage")
    warn("Notebook will fall back to ./local_storage for local dev only")

# ── 4. HuggingFace auth ──────────────────────────────────────
print("\n[4] HuggingFace authentication")
try:
    from huggingface_hub import HfApi, utils as hf_utils
    try:
        api = HfApi()
        user = api.whoami()
        ok(f"Logged in as: {user['name']}")
    except hf_utils.HfHubHTTPError:
        fail("Not authenticated — add huggingface_token to .env or run: huggingface-cli login")
    except Exception as e:
        fail(f"HuggingFace auth error: {e}")
except ImportError:
    fail("huggingface_hub not installed — run pip install cell first")

# ── 5. Dataset repo access ───────────────────────────────────
print("\n[5] Dataset repo access")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    api.dataset_info("deeplearningresearchproject/dataset_project")
    ok("Access confirmed: deeplearningresearchproject/dataset_project")
except Exception as e:
    fail(f"Cannot access dataset repo — {e}")
    warn("Request access at: https://huggingface.co/datasets/deeplearningresearchproject/dataset_project")

# ── 6. Key packages ──────────────────────────────────────────
print("\n[6] Required packages")
packages = {
    "torch":            "PyTorch",
    "torchio":          "TorchIO",
    "nibabel":          "NiBabel",
    "dicom2nifti":      "dicom2nifti",
    "diffusers":        "Diffusers",
    "dotenv":           "python-dotenv",
    "huggingface_hub":  "HuggingFace Hub",
}
for pkg, label in packages.items():
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        ok(f"{label} ({ver})")
    except ImportError:
        fail(f"{label} not installed — re-run pip install cell")

# ── 7. Current dir write access ──────────────────────────────
print("\n[7] Current directory (checkpoint save location)")
cwd = os.getcwd()
test_file = os.path.join(cwd, ".write_test")
try:
    with open(test_file, "w") as f:
        f.write("ok")
    os.remove(test_file)
    free_cwd = shutil.disk_usage(cwd).free / 1e9
    ok(f"Writable: {cwd}  ({free_cwd:.1f} GB free)")
except Exception as e:
    fail(f"Cannot write to current dir: {e}")

# ── Summary ──────────────────────────────────────────────────
print("\n" + "=" * 55)
if all_ok:
    print("  ✅  All checks passed — ready to run!")
else:
    print("  ❌  Fix the issues above before proceeding.")
print("=" * 55)


## 🔧 Install Dependencies

In [ ]:
!pip install diffusers accelerate torchio nibabel dicom2nifti huggingface_hub python-dotenv --quiet

## 📦 Import Libraries

In [ ]:
import os
import zipfile
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchio as tio
import dicom2nifti

from huggingface_hub import login, hf_hub_download
from dotenv import load_dotenv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 🔐 HuggingFace Login

In [ ]:
load_dotenv()
hf_token = os.environ.get("huggingface_token")
login(hf_token)

## 📥 Step: Get Data
> ⚠️ Data is downloaded to  — GPU node local storage — to avoid I/O bottlenecks.  
> Local storage is **temporary**: re-run this cell every time you switch to a new node.

In [ ]:
LOCAL = os.environ.get("LOCAL", "./local_storage")  # fallback for local dev

dicom_dir  = f"{LOCAL}/dataset/dicom"
nifti_dir  = f"{LOCAL}/dataset/nifti"
processed_dir = f"{LOCAL}/dataset/processed"

os.makedirs(dicom_dir, exist_ok=True)
os.makedirs(nifti_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Download zip from HuggingFace Hub into /dataset
filepath = hf_hub_download(
    repo_id="deeplearningresearchproject/dataset_project",
    filename="LIDC-IDRI-0050.zip",
    repo_type="dataset",
    local_dir=f"{LOCAL}/dataset",
)
print(f"Downloaded to: {filepath}")

## 🔄 DICOM → NIfTI Conversion

In [ ]:
class Niftify:
    def __init__(self, zip_path, extract_to, nifti_path):
        self.zip_path = zip_path
        self.extract_to = extract_to
        self.nifti_path = nifti_path

    def unzip_file(self):
        if not os.path.isfile(self.zip_path):
            raise FileNotFoundError(f"ZIP file not found: {self.zip_path}")
        os.makedirs(self.extract_to, exist_ok=True)
        with zipfile.ZipFile(self.zip_path, "r") as zip_ref:
            zip_ref.extractall(self.extract_to)
            print(f"Extracted {len(zip_ref.namelist())} files to '{self.extract_to}'")

    def to_nifti(self):
        assert os.path.isdir(self.extract_to)
        os.makedirs(self.nifti_path, exist_ok=True)
        dicom2nifti.convert_directory(self.extract_to, self.nifti_path)

    def run(self):
        self.unzip_file()
        self.to_nifti()

niftify = Niftify(filepath, dicom_dir, nifti_dir)
niftify.run()
dataset_files = [f for f in os.listdir(nifti_dir) if f.endswith(".nii") or f.endswith(".nii.gz")]
print(f"NIfTI files found: {len(dataset_files)}")

## ⚙️ Preprocessing Pipeline
Resample → CropOrPad to (256×256×128) → MinMax normalize [0, 1]

In [ ]:
target_shape = (256, 256, 128)

pipeline = tio.Compose([
    tio.Resample(1.0),
    tio.CropOrPad(target_shape),
    tio.RescaleIntensity(out_min_max=(0, 1)),
])

for i, fname in enumerate(dataset_files):
    fpath = os.path.join(nifti_dir, fname)
    out_path = os.path.join(processed_dir, fname)
    if os.path.exists(out_path):
        print(f"[{i+1}/{len(dataset_files)}] {fname}: already processed, skipping.")
        continue
    subj = tio.Subject(chest_ct=tio.ScalarImage(fpath))
    print(f"[{i+1}/{len(dataset_files)}] {fname}: {subj.chest_ct.shape}", flush=True)
    trans = pipeline(subj)
    trans.chest_ct.save(out_path)
    del subj, trans

print(f"Preprocessing complete. Files in: {processed_dir}")

## 🗂️ Dataset: 2D Axial Slices
Extract 2D slices (H×W = 256×256) along the axial axis from preprocessed 3D volumes.

In [ ]:
class CTSliceDataset(Dataset):
    """
    Yields individual 2D axial slices from preprocessed 3D CT volumes.
    Each slice shape: (1, 256, 256), values in [0, 1].
    """
    def __init__(self, processed_dir, axis=0, min_content_ratio=0.01):
        self.slices = []
        fnames = [f for f in os.listdir(processed_dir) if f.endswith(".nii") or f.endswith(".nii.gz")]
        for fname in fnames:
            subj = tio.Subject(ct=tio.ScalarImage(os.path.join(processed_dir, fname)))
            vol = subj.ct.data.squeeze().numpy()  # (D, H, W)
            n = vol.shape[axis]
            for i in range(n):
                slc = np.take(vol, i, axis=axis).astype(np.float32)
                # Skip near-empty slices (background only)
                if slc.mean() > min_content_ratio:
                    self.slices.append(slc)
        print(f"Total slices loaded: {len(self.slices)}")

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        slc = self.slices[idx]
        return torch.tensor(slc).unsqueeze(0)  # (1, H, W)


dataset  = CTSliceDataset(processed_dir)
loader   = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

# Sanity check
sample = next(iter(loader))
print(f"Batch shape: {sample.shape}   dtype: {sample.dtype}   range: [{sample.min():.2f}, {sample.max():.2f}]")

## 🧠 VAE — Encoder / Decoder
- Encoder:  → latent  (spatial bottleneck + 512 channels)
- Decoder:  → 
- Latent dimension passed to diffusion U-Net: **512 channels at 16×16**

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
    def forward(self, x): return x + self.net(x)


class VAEEncoder(nn.Module):
    """
    (1, 256, 256) → mu / logvar  shape: (512, 16, 16)
    Downsample ×4 via strided convolutions: 256 → 128 → 64 → 32 → 16
    """
    def __init__(self, latent_ch=512):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1,   64,  3, padding=1),
            ResBlock(64),
            nn.Conv2d(64,  128, 4, stride=2, padding=1),   # 128
            ResBlock(128),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),   # 64
            ResBlock(256),
            nn.Conv2d(256, 512, 4, stride=2, padding=1),   # 32
            ResBlock(512),
            nn.Conv2d(512, 512, 4, stride=2, padding=1),   # 16
            ResBlock(512),
            nn.GroupNorm(8, 512), nn.SiLU(),
        )
        self.to_mu     = nn.Conv2d(512, latent_ch, 1)
        self.to_logvar = nn.Conv2d(512, latent_ch, 1)

    def forward(self, x):
        h = self.enc(x)
        return self.to_mu(h), self.to_logvar(h)


class VAEDecoder(nn.Module):
    """
    (512, 16, 16) → (1, 256, 256)
    Upsample ×4: 16 → 32 → 64 → 128 → 256
    """
    def __init__(self, latent_ch=512):
        super().__init__()
        self.dec = nn.Sequential(
            nn.Conv2d(latent_ch, 512, 3, padding=1),
            ResBlock(512),
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),  # 32
            ResBlock(256),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 64
            ResBlock(128),
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1),  # 128
            ResBlock(64),
            nn.ConvTranspose2d(64,  32,  4, stride=2, padding=1),  # 256
            ResBlock(32),
            nn.GroupNorm(8, 32), nn.SiLU(),
            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z): return self.dec(z)


class VAE(nn.Module):
    def __init__(self, latent_ch=512):
        super().__init__()
        self.encoder = VAEEncoder(latent_ch)
        self.decoder = VAEDecoder(latent_ch)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    @torch.no_grad()
    def encode(self, x):
        mu, _ = self.encoder(x)
        return mu   # use mean as deterministic latent at inference

    @torch.no_grad()
    def decode(self, z):
        return self.decoder(z)


vae = VAE(latent_ch=512).to(device)
print(f"VAE parameters: {sum(p.numel() for p in vae.parameters()):,}")

# Shape sanity check
with torch.no_grad():
    dummy = torch.randn(2, 1, 256, 256).to(device)
    recon, mu, logvar = vae(dummy)
    print(f"Input:   {dummy.shape}")
    print(f"Latent:  {mu.shape}   ← 512-dim feature space at 16×16")
    print(f"Recon:   {recon.shape}")

## 🏋️ Train VAE

In [ ]:
def vae_loss(recon, x, mu, logvar, kl_weight=1e-4):
    recon_loss = F.mse_loss(recon, x)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_weight * kl_loss, recon_loss, kl_loss


VAE_EPOCHS   = 50
VAE_LR       = 1e-4
VAE_CKPT     = "./vae_ckpt.pt"   # current dir — your personal space

vae_opt = torch.optim.AdamW(vae.parameters(), lr=VAE_LR)
vae_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(vae_opt, T_max=VAE_EPOCHS)

vae.train()
for epoch in range(VAE_EPOCHS):
    total_loss = recon_l = kl_l = 0.0
    for batch in loader:
        x = batch.to(device)
        recon, mu, logvar = vae(x)
        loss, rl, kll = vae_loss(recon, x, mu, logvar)

        vae_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        vae_opt.step()

        total_loss += loss.item()
        recon_l    += rl.item()
        kl_l       += kll.item()

    vae_scheduler.step()
    n = len(loader)
    print(f"[VAE] Epoch {epoch+1:>3}/{VAE_EPOCHS}  loss={total_loss/n:.4f}  recon={recon_l/n:.4f}  kl={kl_l/n:.6f}")

torch.save(vae.state_dict(), VAE_CKPT)
print(f"VAE saved → {VAE_CKPT}")

## 🔊 Diffusion U-Net (operates in 512-channel latent space)
Denoises latent tensors of shape  conditioned on timestep embeddings.

In [ ]:
import math

class SinusoidalPosEmb(nn.Module):
    """Timestep → embedding vector of size ."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        )
        args  = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)   # (B, dim)


class TimestepBlock(nn.Module):
    """ResBlock that injects timestep embedding via scale-shift."""
    def __init__(self, ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, ch * 2)  # scale + shift

    def forward(self, x, t_emb):
        scale, shift = self.t_proj(t_emb).chunk(2, dim=1)
        scale = scale[:, :, None, None]
        shift = shift[:, :, None, None]

        h = self.conv1(F.silu(self.norm1(x)))
        h = h * (1 + scale) + shift           # AdaGN-style conditioning
        h = self.conv2(F.silu(self.norm2(h)))
        return x + h


class DiffusionUNet(nn.Module):
    """
    U-Net denoiser for latent tensors (512, 16, 16).
    Channels: 512 → 512 → 256 (bottleneck) → 512 → 512
    """
    def __init__(self, latent_ch=512, t_dim=256):
        super().__init__()
        self.t_emb = nn.Sequential(
            SinusoidalPosEmb(t_dim),
            nn.Linear(t_dim, t_dim * 4), nn.SiLU(),
            nn.Linear(t_dim * 4, t_dim),
        )

        # Encoder arm
        self.enc1 = nn.Conv2d(latent_ch, 512, 3, padding=1)
        self.enc1_tb = TimestepBlock(512, t_dim)

        self.down1 = nn.Conv2d(512, 512, 4, stride=2, padding=1)   # 16→8
        self.enc2_tb = TimestepBlock(512, t_dim)

        # Bottleneck
        self.bot1 = TimestepBlock(512, t_dim)
        self.bot2 = TimestepBlock(512, t_dim)

        # Decoder arm
        self.up1 = nn.ConvTranspose2d(512, 512, 4, stride=2, padding=1)  # 8→16
        self.dec1_tb = TimestepBlock(1024, t_dim)   # skip-cat → 1024

        self.out = nn.Sequential(
            nn.GroupNorm(8, 1024), nn.SiLU(),
            nn.Conv2d(1024, latent_ch, 3, padding=1),
        )

    def forward(self, z, t):
        """
        z: (B, 512, 16, 16)  — noisy latent
        t: (B,)              — integer timesteps
        Returns: (B, 512, 16, 16) — predicted noise
        """
        te = self.t_emb(t)                        # (B, t_dim)

        # Encoder
        e1 = self.enc1(z)                         # (B, 512, 16, 16)
        e1 = self.enc1_tb(e1, te)

        e2 = self.down1(e1)                       # (B, 512,  8,  8)
        e2 = self.enc2_tb(e2, te)

        # Bottleneck
        b  = self.bot1(e2, te)
        b  = self.bot2(b,  te)

        # Decoder
        d1 = self.up1(b)                          # (B, 512, 16, 16)
        d1 = torch.cat([d1, e1], dim=1)           # (B, 1024,16, 16)
        d1 = self.dec1_tb(d1, te)

        return self.out(d1)                       # (B, 512, 16, 16)


unet = DiffusionUNet(latent_ch=512).to(device)
print(f"U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}")

# Shape check
with torch.no_grad():
    z_dummy = torch.randn(2, 512, 16, 16).to(device)
    t_dummy = torch.randint(0, 1000, (2,)).to(device)
    out     = unet(z_dummy, t_dummy)
    print(f"Noise prediction shape: {out.shape}")  # expect (2, 512, 16, 16)

## 📅 DDPM Noise Scheduler

In [ ]:
class DDPMScheduler:
    """
    Linear beta schedule, forward noising, and DDPM reverse sampling.
    """
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.T = T
        betas  = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)

        self.register = lambda name, val: setattr(self, name, val)
        self.betas          = betas
        self.alphas         = alphas
        self.alpha_bar      = alpha_bar
        self.sqrt_ab        = alpha_bar.sqrt()
        self.sqrt_one_m_ab  = (1 - alpha_bar).sqrt()

    def add_noise(self, z0, t):
        """Forward process: z_t = sqrt(ā_t)*z0 + sqrt(1-ā_t)*ε"""
        eps   = torch.randn_like(z0)
        ab    = self.sqrt_ab[t][:, None, None, None]
        s1mab = self.sqrt_one_m_ab[t][:, None, None, None]
        return ab * z0 + s1mab * eps, eps

    @torch.no_grad()
    def sample_step(self, model, z_t, t_scalar):
        """One DDPM reverse step."""
        B = z_t.shape[0]
        t_vec = torch.full((B,), t_scalar, device=z_t.device, dtype=torch.long)

        eps_pred = model(z_t, t_vec)

        beta     = self.betas[t_scalar]
        alpha    = self.alphas[t_scalar]
        ab       = self.alpha_bar[t_scalar]

        # Predicted x0
        coef1 = 1.0 / alpha.sqrt()
        coef2 = beta / (1 - ab).sqrt()
        z_prev = coef1 * (z_t - coef2 * eps_pred)

        if t_scalar > 0:
            z_prev = z_prev + beta.sqrt() * torch.randn_like(z_t)
        return z_prev


scheduler = DDPMScheduler(T=1000, device=device)
print("DDPM scheduler ready.")

## 🏋️ Train Diffusion Model (LDM)
First encode all slices to latent codes, then train the U-Net to predict noise.

In [ ]:
# ── Load VAE if needed ──────────────────────────────────────────
vae.load_state_dict(torch.load(VAE_CKPT, map_location=device))
vae.eval()
print("VAE loaded.")

# ── Pre-encode all slices to latents ────────────────────────────
print("Encoding dataset to latent space ...")
vae.eval()
all_latents = []
with torch.no_grad():
    for batch in loader:
        x = batch.to(device)
        z = vae.encode(x)            # (B, 512, 16, 16)
        all_latents.append(z.cpu())
all_latents = torch.cat(all_latents, dim=0)
print(f"Latent bank: {all_latents.shape}  — {all_latents.element_size()*all_latents.nelement()/1e6:.1f} MB")

# ── Latent dataset ───────────────────────────────────────────────
from torch.utils.data import TensorDataset

latent_ds     = TensorDataset(all_latents)
latent_loader = DataLoader(latent_ds, batch_size=8, shuffle=True, pin_memory=True)

# ── Training ─────────────────────────────────────────────────────
DIFF_EPOCHS = 100
DIFF_LR     = 2e-4
DIFF_CKPT   = "./diffusion_ckpt.pt"   # current dir — your personal space

diff_opt  = torch.optim.AdamW(unet.parameters(), lr=DIFF_LR)
diff_sched = torch.optim.lr_scheduler.CosineAnnealingLR(diff_opt, T_max=DIFF_EPOCHS)

unet.train()
for epoch in range(DIFF_EPOCHS):
    total = 0.0
    for (z0,) in latent_loader:
        z0 = z0.to(device)
        B  = z0.shape[0]

        # Random timesteps
        t  = torch.randint(0, scheduler.T, (B,), device=device)

        # Forward noising
        z_t, eps = scheduler.add_noise(z0, t)

        # Predict noise
        eps_pred = unet(z_t, t)
        loss     = F.mse_loss(eps_pred, eps)

        diff_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        diff_opt.step()
        total += loss.item()

    diff_sched.step()
    print(f"[DIFF] Epoch {epoch+1:>3}/{DIFF_EPOCHS}  noise_mse={total/len(latent_loader):.5f}")

torch.save(unet.state_dict(), DIFF_CKPT)
print(f"Diffusion model saved → {DIFF_CKPT}")

## 🎨 Generate New CT Slices

In [ ]:
@torch.no_grad()
def generate_slices(n=4, T=1000):
    """Sample n CT slices from pure Gaussian noise."""
    unet.eval()
    vae.eval()

    # Start from noise in latent space
    z = torch.randn(n, 512, 16, 16, device=device)

    # Reverse diffusion
    for t in reversed(range(T)):
        z = scheduler.sample_step(unet, z, t)

    # Decode latents → pixel space
    imgs = vae.decode(z)            # (n, 1, 256, 256)  values in [0,1]
    return imgs.cpu()


generated = generate_slices(n=4)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(generated[i, 0], cmap="gray")
    ax.set_title(f"Generated #{i+1}")
    ax.axis("off")
plt.suptitle("LDM — Generated CT Slices (512-dim latent space)", fontsize=13)
plt.tight_layout()
plt.show()

## 📊 Reconstruction Quality Check
Compare real slices vs VAE reconstructions to verify the latent space quality.

In [ ]:
vae.eval()
with torch.no_grad():
    real_batch = next(iter(loader))[:4].to(device)
    recon_batch, _, _ = vae(real_batch)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(real_batch[i, 0].cpu(), cmap="gray")
    axes[0, i].set_title(f"Real #{i+1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_batch[i, 0].cpu(), cmap="gray")
    axes[1, i].set_title(f"Recon #{i+1}")
    axes[1, i].axis("off")

plt.suptitle("VAE: Real vs Reconstruction", fontsize=13)
plt.tight_layout()
plt.show()